In [139]:
import numpy as np
import pandas as pd
from sympy import limit
import wandb

api = wandb.Api(
    api_key="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",
    timeout=60
)

# 1. Get last 10 runs (sorted by creation time descending)
runs = api.runs(
    "eibl-usc/graph-clip",
    filters={
        # "display_name": {"$regex": "trained_on_.._eval_on_.._..?_shot_.+"},
        # "config.dataset": "ukr_rus_twitter",
        # "config.task_name": "neighbor_matching",
    },
    order="-created_at",
    per_page=10,
    # limit to 10:
    lazy=False,
)

In [ ]:
rows = []
for run in runs:
    attrs = getattr(run, "_attrs", {}) or {}
    params = ((attrs.get("config") or {}).get("params") or {})
    summary = attrs.get("summaryMetrics") or {}

    rows.append({
        "run_id": attrs.get("name"),
        "display_name": attrs.get("displayName"),
        "state": attrs.get("state"),
        "dataset": params.get("dataset"),
        "task_name": params.get("task_name"),
        "prefix": params.get("prefix"),
        "pretrained_model_run": params.get("pretrained_model_run"),
        "n_shots": params.get("n_shots"),
        "n_way": params.get("n_way"),
        "n_query": params.get("n_query"),
        "zero_shot": params.get("zero_shot"),
        "test_accuracy": summary.get("test_accuracy"),
        "train_accuracy": summary.get("train_accuracy"),
        "test_f1": summary.get("test_f1"),
        "test_roc_auc": summary.get("test_roc_auc"),
        "created_at": attrs.get("createdAt"),
        'steps': attrs['historyKeys']['keys'].get('_step', {}).get('typeCounts', [{}])[0].get('count', np.nan),
        'tags': attrs.get("tags", []),
    })
    
    


In [223]:
df = pd.DataFrame(rows)
df["train1_dataset"] = df["pretrained_model_run"].str.extract(r"train1_(ukr_rus_twitter|midterm|covid19_twitter|election20202|covid_political|ukr_rus_suspended)_")
df["train1_task"] = df["pretrained_model_run"].str.extract(r"train1_.+?_(nm|pl|lp)_")
df["eval1_task"] = df["task_name"].map({
    "neighbor_matching": "nm",
    "temporal_link_prediction": "lp",
    "classification": "pl",
})
df['created_at'] = pd.to_datetime(df['created_at'])
df["shot_label"] = df.apply(lambda r: 0 if bool(r.get("zero_shot", False)) else r["n_shots"], axis=1)
# df['is_eval'] = df['display_name'].str.contains(r"eval")
# plot_df = df[df["eval1_task"].isin(EVAL_TASKS) & df["train1_task"].eq("nm")].copy()
df['eval1_dataset'] = df['dataset']
df['trained_on_display_name'] = df.pretrained_model_run.str.split('/').str[1]
df['month/day'] = df['created_at'].dt.month.astype(str) + '/' + df['created_at'].dt.day.astype(str)
df = df.sort_values('created_at', ascending=False)
d = {
    'trained_on_n_way': 'n_way',
    'trained_on_n_query': 'n_query',
    'trained_on_n_shots': 'n_shots',
    'trained_on_steps': 'steps',
}
mask = df['trained_on_display_name'].isin(df['display_name'])
existing_trained_on_display_names = df.trained_on_display_name[mask]
dname2stats = df.set_index('display_name').loc[existing_trained_on_display_names][list(d.values())]
for col, stat in d.items():
    df[col] = df.trained_on_display_name.map(dname2stats[stat].to_dict())
df__ = df.copy()

# df = df__[df__.tags.astype(str).str.contains('social_llm_v4', regex=False, na=False)]

t1 = df__[df__.tags.astype(str).str.contains('v4')].created_at.tolist()

start, end = min(t1), max(t1)

mask = df__.created_at.between(start, end)

df__.loc[mask, 'tags'] = df__.loc[mask, 'tags'].apply(lambda x: x + ['social_llm_v4'])

df = df__[df__.tags.astype(str).str.contains('social_llm_v4', regex=False, na=False)]

In [182]:
unique_on = ['train1_dataset', 'train1_task', 'eval1_dataset', 'eval1_task', 'n_shots']

In [263]:
df["seq_id"] = df.train1_dataset.fillna("NA").astype(str) + "+" + df.train1_task.fillna("NA").astype(str) + "|" + df.eval1_dataset.fillna("NA").astype(str) + "+" + df.eval1_task.fillna("NA").astype(str) + "(" + df.n_shots.astype(str) + ")"

In [264]:
# df['train_id'] = df.seq_id.str.split(">").str[1]
# df['train_id']
# df.groupby('seq_id').n_shots.value_counts()
# df.groupby(unique_on).test_roc_auc.value_counts()
df['auc_rounded'] = df.test_roc_auc.round(4)
df = df.sort_values('created_at', ascending=False).drop_duplicates(unique_on + ['auc_rounded'], keep='first')

In [266]:
df.seq_id

0      covid_political+pl|election2020+pl(10.0)
1      covid_political+pl|election2020+nm(10.0)
2       covid_political+pl|election2020+pl(1.0)
3       covid_political+pl|election2020+nm(1.0)
4       covid_political+pl|election2020+pl(0.0)
                         ...                   
194        midterm+nm|ukr_rus_suspended+pl(0.0)
195          midterm+nm|covid_political+nm(0.0)
196             midterm+nm|election2020+nm(0.0)
197        midterm+nm|ukr_rus_suspended+nm(0.0)
198    ukr_rus_twitter+pl|election2020+pl(10.0)
Name: seq_id, Length: 199, dtype: str

In [269]:
# df.groupby(unique_on)['test_roc_auc'].value_counts()
for dataset1 in df.train1_dataset.unique():
    for task1 in df.train1_task.unique():
        for dataset2 in df.train1_dataset.unique():
            for task2 in df.train1_task.unique():
                for n_shots in df.n_shots.unique():
                    id = f'{dataset1}+{task1}|{dataset2}+{task2}({n_shots})'
                    if task1 == 'pl':
                        continue
                    if dataset1 == dataset2 and task1 == task2:
                        continue
                    if df.seq_id.eq(id).sum() != 1:
                        print(id)

covid_political+nm|covid_political+lp(10.0)
covid_political+nm|covid_political+lp(1.0)
covid_political+nm|covid_political+lp(0.0)
covid_political+nm|covid19_twitter+pl(10.0)
covid_political+nm|covid19_twitter+pl(1.0)
covid_political+nm|covid19_twitter+pl(0.0)
covid_political+nm|covid19_twitter+nm(10.0)
covid_political+nm|covid19_twitter+nm(1.0)
covid_political+nm|covid19_twitter+nm(0.0)
covid_political+nm|covid19_twitter+lp(10.0)
covid_political+nm|covid19_twitter+lp(1.0)
covid_political+nm|covid19_twitter+lp(0.0)
covid_political+nm|ukr_rus_twitter+pl(10.0)
covid_political+nm|ukr_rus_twitter+pl(1.0)
covid_political+nm|ukr_rus_twitter+pl(0.0)
covid_political+nm|ukr_rus_twitter+nm(10.0)
covid_political+nm|ukr_rus_twitter+nm(1.0)
covid_political+nm|ukr_rus_twitter+nm(0.0)
covid_political+nm|ukr_rus_twitter+lp(10.0)
covid_political+nm|ukr_rus_twitter+lp(1.0)
covid_political+nm|ukr_rus_twitter+lp(0.0)
covid_political+nm|midterm+pl(10.0)
covid_political+nm|midterm+pl(1.0)
covid_political+nm|